### Import Packages

In [64]:
import requests
import pandas as pd
import json
import numpy as np
import os

def load_config(config_path='config.json'):
    """
    Load configuration from a JSON file.

    Args:
        config_path (str): Path to the configuration file.

    Returns:
        dict: Dictionary containing configuration variables.
    """
    try:
        with open(config_path, 'r') as file:
            config = json.load(file)
            return config
    except FileNotFoundError:
        print(f"Configuration file not found: {config_path}")
        exit(1)  # Exit with error code
    except json.JSONDecodeError as e:
        print(f"Error parsing configuration file: {e}")
        exit(1)

# Load configuration
config = load_config()

# Access variables
folder = config.get("folder")  # Default folder
draft_league_id = config.get("draft_league_id", None)  # No default; must be set


### Create the base table for the data with one row per GW and per Entry

In [65]:
def get_draft_league_data(draft_league_id):
    """
    Fetches and processes data for a draft league from the Premier League API.

    Parameters:
        draft_league_id (int): The ID of the draft league.

    Returns:
        pd.DataFrame: A DataFrame containing gameweeks and team information.
    """
    # API endpoint
    draft_league_url = f'https://draft.premierleague.com/api/league/{draft_league_id}/details'

    # Fetch data from the API
    response = requests.get(draft_league_url)
    response.raise_for_status()  # Raise an error for bad HTTP status codes
    data = response.json()

    # Process team data
    teams_df = pd.DataFrame(data["league_entries"])
    teams = teams_df.drop(columns=["joined_time"])
    teams['entry_name'] = teams['entry_name'].fillna('Average')

    # Sort and add alphabetic rank
    teams = teams.sort_values('entry_name').reset_index(drop=True)
    teams['alphabetic_order'] = teams.index + 1  # Rank starts at 1

    # Prepare gameweeks DataFrame
    gw = pd.DataFrame({'gw': range(1, 39)})
    teams['key'] = 1
    gw['key'] = 1

    # Merge gameweeks with teams
    teams['entry_id'] = teams['entry_id'].astype(str).str.replace('.0', '', regex=False)
    gw_and_team = pd.merge(gw, teams, on='key').drop('key', axis=1)

    # Rename columns for clarity
    gw_and_team = gw_and_team.rename(columns={
        "id": "team",
        "entry_name": "team_name",
        "player_first_name": "team_player_first_name",
        "player_last_name": "team_player_last_name"
    })

     # Fetch player stats
    PLAYER_STATS_URL = 'https://draft.premierleague.com/api/bootstrap-static'
    player_stats_response = requests.get(PLAYER_STATS_URL)
    player_stats_response.raise_for_status()
    player_stats_data = player_stats_response.json()

    # Process player metadata
    elements_base = pd.DataFrame(player_stats_data["elements"])
    elements_base = elements_base[['id']]
    elements_base['id'] = elements_base['id'].astype(str)
    elements_base = elements_base.rename(columns={"id": "element_id"})

    elements_base['key'] = 1

    # Merge gameweeks with teams
    gw_and_element = pd.merge(gw, elements_base, on='key').drop('key', axis=1)

    return gw_and_team, teams, gw_and_element

### Create Dataframe for all matches and results

In [66]:
def process_match_results(draft_league_id, teams):
    """
    Fetches and processes match results for a draft league from the Premier League API.

    Parameters:
        draft_league_id (int): The ID of the draft league.
        teams (pd.DataFrame): A DataFrame containing team information with IDs and names.

    Returns:
        pd.DataFrame: A DataFrame containing detailed match results with cumulative stats and rankings.
    """
    # API endpoint
    draft_league_url = f'https://draft.premierleague.com/api/league/{draft_league_id}/details'

    # Fetch data from the API
    response = requests.get(draft_league_url)
    response.raise_for_status()  # Raise an error for bad HTTP status codes
    data = response.json()

    # Process match data
    matches_df = pd.DataFrame(data["matches"])
    matches = matches_df.drop(columns=["winning_league_entry", "winning_method"])

    # Define outcomes for each team
    conditions_team_1 = [
        (matches['league_entry_1_points'] > matches['league_entry_2_points']),
        (matches['league_entry_1_points'] < matches['league_entry_2_points']),
        (matches['league_entry_1_points'] == matches['league_entry_2_points'])
    ]
    outcomes_team_1 = ['Win', 'Lose', 'Draw']

    conditions_team_2 = [
        (matches['league_entry_2_points'] > matches['league_entry_1_points']),
        (matches['league_entry_2_points'] < matches['league_entry_1_points']),
        (matches['league_entry_2_points'] == matches['league_entry_1_points'])
    ]
    outcomes_team_2 = ['Win', 'Lose', 'Draw']

    matches['league_entry_1_result'] = np.select(conditions_team_1, outcomes_team_1, default='Unknown')
    matches['league_entry_2_result'] = np.select(conditions_team_2, outcomes_team_2, default='Unknown')

    # Reshape the data for team 1 and team 2
    df_team1 = pd.DataFrame({
        'gw': matches['event'],
        'finished': matches['finished'],
        'team': matches['league_entry_1'],
        'opponent': matches['league_entry_2'],
        'team_points': matches['league_entry_1_points'],
        'opponent_points': matches['league_entry_2_points'],
        'team_result': matches['league_entry_1_result']
    })

    df_team2 = pd.DataFrame({
        'gw': matches['event'],
        'finished': matches['finished'],
        'team': matches['league_entry_2'],
        'opponent': matches['league_entry_1'],
        'team_points': matches['league_entry_2_points'],
        'opponent_points': matches['league_entry_1_points'],
        'team_result': matches['league_entry_2_result']
    })

    # Concatenate data for all teams
    final_df = pd.concat([df_team1, df_team2], ignore_index=True)
    final_df["points_difference"] = final_df["team_points"] - final_df["opponent_points"]
    final_df['result_points'] = final_df['team_result'].map({'Win': 3, 'Lose': 0, 'Draw': 1})

    # Merge with team details
    detailed_final_df = pd.merge(final_df, teams, left_on="team", right_on="id")
    full_detailed_final_df = pd.merge(detailed_final_df, teams, left_on="opponent", right_on="id")

    # Rename columns for clarity
    match_results = full_detailed_final_df.rename(columns={
        "entry_name_x": "team_name",
        "player_first_name_x": "team_player_first_name",
        "player_last_name_x": "team_player_last_name",
        "entry_name_y": "opponent_name",
        "player_first_name_y": "opponent_player_first_name",
        "player_last_name_y": "opponent_player_last_name"
    })

    # Select and sort relevant columns
    match_results = match_results[[
        "gw", "finished", "team", "team_name", "team_player_first_name",
        "team_player_last_name", "opponent", "opponent_name", "opponent_player_first_name", 
        "opponent_player_last_name", "team_points", "opponent_points", "points_difference",
        "team_result", "result_points"
    ]]

    # Calculate cumulative totals
    match_results = match_results.sort_values(by=["gw"], ascending=True)
    match_results['cumulative_team_points'] = match_results.groupby('team')['team_points'].cumsum()
    match_results['cumulative_opponent_points'] = match_results.groupby('team')['opponent_points'].cumsum()
    match_results['cumulative_result_points'] = match_results.groupby('team')['result_points'].cumsum()

    # Sort and rank teams
    match_results_ordered = match_results.sort_values(
        by=['gw', 'cumulative_result_points', 'cumulative_team_points', 'team_name'],
        ascending=[True, False, False, True]
    )
    match_results_ordered["league_position"] = match_results_ordered.groupby('gw').cumcount() + 1
    match_results_ordered['gw_team_points_rank'] = match_results_ordered.groupby(['gw'])['team_points'].rank(ascending=False)
    match_results_ordered['gw_opponent_points_rank'] = match_results_ordered.groupby(['gw'])['opponent_points'].rank(ascending=False)

    return match_results_ordered


### Pull Data for Player Stats for each GW

In [67]:
def fetch_player_stats(gw_and_element):
    """
    Fetches player statistics and gameweek data from the Draft Premier League API,
    processes it, and returns a DataFrame containing player statistics per gameweek.

    Returns:
        pd.DataFrame: A DataFrame with aggregated player statistics and metadata.
    """
    # Fetch player stats
    PLAYER_STATS_URL = 'https://draft.premierleague.com/api/bootstrap-static'
    player_stats_response = requests.get(PLAYER_STATS_URL)
    player_stats_response.raise_for_status()
    player_stats_data = player_stats_response.json()

    # Process player metadata
    elements_df = pd.DataFrame(player_stats_data["elements"])
    elements_df_tidy = elements_df[['id', 'code', 'first_name', 'second_name', 'web_name', 'element_type', 'team']]
    elements_df_tidy['id'] = elements_df_tidy['id'].astype(str)

    # Fetch gameweek stats
    all_gw_data = []
    for gw in range(1, 39):
        GW_SCORES_URL = f'https://draft.premierleague.com/api/event/{gw}/live'
        gw_stats_response = requests.get(GW_SCORES_URL)
        gw_stats_response.raise_for_status()
        gw_stats_data = gw_stats_response.json()
        
        # Append gameweek data
        all_gw_data.append({
            "gw": gw,
            "data": gw_stats_data
        })

    # Extract and process gameweek data
    extracted_data = []
    for item in all_gw_data:
        current_gw = item.get('gw')
        elements = item['data'].get('elements', {})

        for element_id, element_info in elements.items():
            explain_list = element_info.get('explain', [])
            for explain in explain_list:
                for entry in explain[0]:
                    extracted_data.append({
                        'gw': current_gw,
                        'number': element_id,
                        'name': entry['name'],
                        'points': entry['points'],
                        'value': entry['value'],
                        'stat': entry['stat']
                    })

    # Create a DataFrame if data was extracted
    if not extracted_data:
        raise ValueError("No data extracted from the API.")

    player_stats_by_gw = pd.DataFrame(extracted_data)

    # Ensure column types are consistent
    player_stats_by_gw['number'] = player_stats_by_gw['number'].astype(str)
    

    # Merge metadata with gameweek stats
    player_stats_by_gw = pd.merge(player_stats_by_gw, elements_df_tidy, left_on='number', right_on='id')

    # Add a 'played' column to indicate if the player participated
    player_stats_by_gw['played'] = player_stats_by_gw.apply(
        lambda row: 1 if row['stat'] == 'minutes' and row['value'] > 0 else 0, axis=1
    )

    # Group by gameweek and player, aggregating stats
    grouped_player_stats_by_gw = (
        player_stats_by_gw.groupby(['gw', 'number'])
        .agg(
            total_points=('points', 'sum'),
            minutes_played=('played', 'max')
        )
        .reset_index()
    )

    # Replace binary played values with "yes"/"no"
    grouped_player_stats_by_gw['minutes_played'] = grouped_player_stats_by_gw['minutes_played'].apply(
        lambda x: "yes" if x > 0 else "no"
    )

    grouped_player_stats_by_gw = pd.merge(gw_and_element, grouped_player_stats_by_gw, how='left', left_on=['gw', 'element_id'], right_on=['gw', 'number'])
    grouped_player_stats_by_gw = grouped_player_stats_by_gw.drop(columns= ['number'])
    grouped_player_stats_by_gw = grouped_player_stats_by_gw.rename(columns= {'element_id' : 'number'})

    return grouped_player_stats_by_gw, elements_df_tidy, elements_df


### Team LineUps by Manager & Player Scores by GW

In [89]:
def fetch_team_and_player_scores(teams, grouped_player_stats_by_gw, elements_df_tidy):
    """
    Fetches team line-ups and player scores, processes the data, and returns a detailed DataFrame.

    Parameters:
        teams (pd.DataFrame): DataFrame containing team metadata.
        grouped_player_stats_by_gw (pd.DataFrame): DataFrame with player stats grouped by gameweek.
        elements_df_tidy (pd.DataFrame): DataFrame with player metadata.

    Returns:
        pd.DataFrame: A DataFrame containing detailed player scores with team and cost information.
    """
    # Filter out NaN values from 'entry_id' and process the list of team IDs
    teams = pd.DataFrame(teams)
    filtered_teams = teams[teams['entry_id'].notna()]
    filtered_teams_list = filtered_teams['entry_id'].astype(str).tolist()

    # Collect data for each team's gameweek line-ups
    team_data = []
    for team_id in filtered_teams_list:
        for gw in range(1, 39):  # Gameweeks 1 through 38
            TEAM_LINEUPS_URL = f'https://draft.premierleague.com/api/entry/{team_id}/event/{gw}'
            response = requests.get(TEAM_LINEUPS_URL)
            if response.status_code == 200:
                gw_data = response.json()
                team_data.append({
                    'team_id': team_id,
                    'gw': gw,
                    'picks': gw_data.get('picks', [])
                })
                print(f"Fetched data for Team ID {team_id} and GW {gw}: {response.status_code}")
            else:
                print(f"Failed to fetch data for Team ID {team_id} and GW {gw}: {response.status_code}")

    # Process team data into a DataFrame
    team_data_df = pd.DataFrame(team_data)
    exploded_df = team_data_df.explode('picks').reset_index(drop=True)
    picks_df = pd.json_normalize(exploded_df['picks'])

    # Combine picks with team and gameweek metadata
    team_line_up = pd.concat([exploded_df[['team_id', 'gw']], picks_df], axis=1)
    team_line_up = team_line_up[['team_id', 'gw', 'element', 'position']]
    team_line_up['element'] = team_line_up['element'].astype(str)

    # Merge player stats with team line-ups
    team_points_by_gw = pd.merge(
        grouped_player_stats_by_gw, 
        team_line_up, 
        left_on=['gw', 'number'], 
        right_on=['gw', 'element'], 
        how='left'
    )
    team_points_by_gw = team_points_by_gw[['team_id', 'gw', 'number', 'position', 'total_points', 'minutes_played']]

    # Merge with player metadata
    team_points_by_gw_detailed = pd.merge(
        team_points_by_gw, 
        elements_df_tidy, 
        left_on='number', 
        right_on='id'
    )
    team_points_by_gw_detailed = team_points_by_gw_detailed[[
        'team_id', 'gw', 'number', 'position', 'total_points', 'minutes_played',
        'first_name', 'second_name', 'web_name', 'element_type', 'team'
    ]]
    team_points_by_gw_detailed['status'] = np.where(
        team_points_by_gw_detailed['position'] <= 11, 'played', 'benched'
    )

    # Add team and player details
    team_player_scores = pd.merge(
        team_points_by_gw_detailed, 
        teams, 
        left_on='team_id', 
        right_on='entry_id', 
        how='left'
    )
    team_player_scores = team_player_scores[[
        'team_id', 'entry_name', 'player_first_name', 'player_last_name', 'gw',
        'number', 'position', 'total_points', 'minutes_played', 'first_name',
        'second_name', 'web_name', 'element_type', 'team', 'status'
    ]]

    # Fetch gameweek prices
    players_url = "https://fantasy.premierleague.com/api/bootstrap-static/"
    response = requests.get(players_url)
    response.raise_for_status()
    players = response.json()['elements']

    gw_prices = []
    for player in players:
        player_id = player['id']
        player_name = f"{player['first_name']} {player['second_name']}"
        player_history_url = f"https://fantasy.premierleague.com/api/element-summary/{player_id}/"
        player_response = requests.get(player_history_url)
        player_data = player_response.json()
        history = player_data.get('history', [])
        for record in history:
            gw_prices.append({
                "Player ID": str(player_id),
                "Player Name": player_name,
                "GW": record['round'],
                "Price": record['value'] / 10  # Convert to millions
            })
            print(f"Fetched data for Player {player_name} and GW {record['round']}: {response.status_code}")

    # Merge player prices into team scores
    gw_prices_df = pd.DataFrame(gw_prices)
    team_player_scores_with_cost = pd.merge(
        team_player_scores, 
        gw_prices_df, 
        left_on=['number', 'gw'], 
        right_on=['Player ID', 'GW'], 
        how='left'
    ).drop(columns=['Player ID', 'GW'])

    team_player_scores_with_cost['Price'] = team_player_scores_with_cost.groupby('number')['Price'].ffill()
    team_player_scores_with_cost['Player Name'] = team_player_scores_with_cost.groupby('number')['Player Name'].ffill()
    team_player_scores_with_cost = team_player_scores_with_cost.dropna(subset = ['team_id'])

    team_player_scores_with_cost['Price'] = team_player_scores_with_cost.groupby('number')['Price'].ffill()
    team_player_scores_with_cost['Player Name'] = team_player_scores_with_cost.groupby('number')['Player Name'].ffill()
    team_player_scores_with_cost = team_player_scores_with_cost.dropna(subset = ['team_id'])
    team_player_scores_with_cost['minutes_played'] = team_player_scores_with_cost['minutes_played'].fillna('no')
    team_player_scores_with_cost['total_points'] = team_player_scores_with_cost['total_points'].fillna(0)

    return team_player_scores_with_cost

# Example usage
# teams_df = fetch_teams()
# grouped_player_stats = fetch_player_stats()
# elements_df = fetch_elements()
# detailed_scores = fetch_team_and_player_scores(teams_df, grouped_player_stats, elements_df)


### Originally Drafted Teams

In [69]:
def process_draft_picks(draft_league_id, elements_df):
    """
    Processes draft picks for a given draft league, calculates rankings and comparisons,
    and returns a DataFrame with enriched draft pick data.

    Parameters:
        draft_league_id (int): The ID of the draft league.
        elements_df (pd.DataFrame): A DataFrame containing player metadata.

    Returns:
        pd.DataFrame: A DataFrame with enriched draft pick data, including rankings and comparisons.
    """
    # Fetch draft choices data from the API
    CHOICES_URL = f'https://draft.premierleague.com/api/draft/{draft_league_id}/choices'
    choices_response = requests.get(CHOICES_URL)
    choices_response.raise_for_status()  # Raise an error for bad HTTP status codes
    choices_data = choices_response.json()

    # Create a DataFrame from the API data
    choices_df = pd.DataFrame(choices_data["choices"])
    choices_df['element'] = choices_df['element'].astype(str)
    elements_df['id'] = elements_df['id'].astype(str)

    # Merge draft picks with player metadata
    draft_picks_df = pd.merge(
        choices_df, 
        elements_df, 
        how='right', 
        left_on='element', 
        right_on='id'
    )

    # Calculate rankings and draft metrics
    draft_picks_df['position_points_rank'] = (
        draft_picks_df[draft_picks_df['entry_name'].notna()]  # Filter out rows with NaN in 'entry_name'
        .groupby(['element_type'])['total_points']
        .rank(method='first', ascending=False)
    )
    draft_picks_df['pick_order'] = (draft_picks_df['round'] - 1) * 9 + draft_picks_df['pick']
    draft_picks_df['position_pick_order'] = draft_picks_df.groupby(['element_type'])['pick_order'].rank(ascending=True)
    draft_picks_df['points_vs_position_pick'] = draft_picks_df['position_pick_order'] - draft_picks_df['position_points_rank']
    draft_picks_df['points_vs_position_pick_rank'] = draft_picks_df.groupby(['element_type'])['points_vs_position_pick'].rank(
        method='first', ascending=False
    )
    draft_picks_df['all_players_position_points_rank'] = (
        draft_picks_df.groupby(['element_type'])['total_points'].rank(method='first', ascending=False)
    )
    draft_picks_df['all_players_points_vs_position_pick'] = (
        draft_picks_df['position_pick_order'] - draft_picks_df['all_players_position_points_rank']
    )
    draft_picks_df['all_players_points_vs_position_pick_rank'] = draft_picks_df.groupby(['element_type'])[
        'all_players_points_vs_position_pick'
    ].rank(method='first', ascending=False)

    return draft_picks_df


### Transfers Made

In [70]:
def get_transfers(DRAFT_LEAGUE_ID, elements_df_tidy, team_player_scores_with_cost):
    # Construct URL with the provided DRAFT_LEAGUE_ID
    TRANSACTIONS_URL = f'https://draft.premierleague.com/api/draft/league/{DRAFT_LEAGUE_ID}/transactions'
    
    # Make the GET request
    transfers_response = requests.get(TRANSACTIONS_URL)
    
    # Raise an error if the request was not successful
    transfers_response.raise_for_status()
    
    # Parse the response JSON and load into a DataFrame
    transfers_data = transfers_response.json()
    transfers_df = pd.DataFrame(transfers_data["transactions"])
    
    # Ensure the columns are treated as strings
    transfers_df['element_in'] = transfers_df['element_in'].astype(str)
    transfers_df['element_out'] = transfers_df['element_out'].astype(str)
    
    # Merge data with the elements_df_tidy DataFrame for both 'element_in' and 'element_out'
    transfers_df1 = pd.merge(transfers_df, elements_df_tidy, left_on='element_in', right_on='id')
    transfers_df2 = pd.merge(transfers_df1, elements_df_tidy, left_on='element_out', right_on='id')
    
    # Select the relevant columns to return
    transfers = transfers_df2[['element_in', 'element_out', 'entry', 'event', 'kind', 'result', 'web_name_x', 'web_name_y']]
    transfers['id'] = range(1, len(transfers) + 1)

    player_scores_by_gw_transfers = team_player_scores_with_cost[['gw','number', 'total_points', 'minutes_played']]
    transfers_with_scores = pd.merge(transfers, player_scores_by_gw_transfers, how='left', left_on='element_in', right_on='number')
    transfers_with_scores = transfers_with_scores[transfers_with_scores['gw'] >= transfers_with_scores['event']] 
    transfers_with_scores = transfers_with_scores.groupby(['id', 'element_in', 'element_out', 'entry', 'event', 'kind', 'result', 'web_name_x', 'web_name_y'], as_index=False).agg(total_points_in = ('total_points', 'sum'))
    transfers_with_scores_both = pd.merge(transfers_with_scores, player_scores_by_gw_transfers, how='left', left_on='element_out', right_on='number')
    transfers_with_scores_both = transfers_with_scores_both[transfers_with_scores_both['gw'] >= transfers_with_scores_both['event']] 
    transfers = transfers_with_scores_both.groupby(['id', 'element_in', 'element_out', 'entry', 'event', 'kind', 'result', 'web_name_x', 'web_name_y', 'total_points_in'], as_index=False).agg(total_points_out = ('total_points', 'sum'))

    
    
    return transfers


### Display all Data

In [71]:
gw_and_team, teams, gw_and_element = get_draft_league_data(draft_league_id)

match_results_ordered = process_match_results(draft_league_id, teams)

grouped_player_stats_by_gw, elements_df_tidy, elements_df = fetch_player_stats(gw_and_element)

team_player_scores_with_cost = fetch_team_and_player_scores(teams, grouped_player_stats_by_gw, elements_df_tidy)

draft_picks_df = process_draft_picks(draft_league_id, elements_df)

transfers = get_transfers(draft_league_id, elements_df_tidy, team_player_scores_with_cost)



C:\Users\scottrenfreetuck\AppData\Local\Temp\ipykernel_23564\2125902896.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  elements_df_tidy['id'] = elements_df_tidy['id'].astype(str)


Fetched data for Team ID 289293 and GW 1: 200
Fetched data for Team ID 289293 and GW 2: 200
Fetched data for Team ID 289293 and GW 3: 200
Fetched data for Team ID 289293 and GW 4: 200
Fetched data for Team ID 289293 and GW 5: 200
Fetched data for Team ID 289293 and GW 6: 200
Fetched data for Team ID 289293 and GW 7: 200
Fetched data for Team ID 289293 and GW 8: 200
Fetched data for Team ID 289293 and GW 9: 200
Fetched data for Team ID 289293 and GW 10: 200
Fetched data for Team ID 289293 and GW 11: 200
Fetched data for Team ID 289293 and GW 12: 200
Fetched data for Team ID 289293 and GW 13: 200
Fetched data for Team ID 289293 and GW 14: 200
Fetched data for Team ID 289293 and GW 15: 200
Failed to fetch data for Team ID 289293 and GW 16: 404
Failed to fetch data for Team ID 289293 and GW 17: 404
Failed to fetch data for Team ID 289293 and GW 18: 404
Failed to fetch data for Team ID 289293 and GW 19: 404
Failed to fetch data for Team ID 289293 and GW 20: 404
Failed to fetch data for Team

C:\Users\scottrenfreetuck\AppData\Local\Temp\ipykernel_23564\1054754086.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transfers['id'] = range(1, len(transfers) + 1)


In [72]:
display(gw_and_team)
display(match_results_ordered)
display(team_player_scores_with_cost)
display(transfers)
display(draft_picks_df)

,gw,entry_id,team_name,team,team_player_first_name,team_player_last_name,short_name,waiver_pick,alphabetic_order
0,1,289293,*Stratford Sardines*,291619,Flat,7,F7,2.0,1
1,1,nan,Average,336507,None,None,AV,NaN,2
2,1,228518,Boom Boom Eagle FC,229876,Sam,Thorpe,ST,9.0,3
3,1,283372,CripplingHannover96,285674,Ryan,McIntyre,RM,5.0,4
4,1,184585,Easy Three Points,185528,Jonathan,Ako,JA,6.0,5
...,...,...,...,...,...,...,...,...,...
375,38,324701,Eleven Hags,327410,Liam,Thaker,LT,3.0,6
376,38,184679,Kernowfornia United,185622,Scott,Renfree-Tuck,SR,8.0,7
377,38,317751,Some Solanke-Panky?,320432,Alex,Turnbull,AT,4.0,8
378,38,325685,TAA very much,328396,James,Mason,JM,7.0,9


,gw,finished,team,team_name,team_player_first_name,team_player_last_name,opponent,opponent_name,opponent_player_first_name,opponent_player_last_name,...,opponent_points,points_difference,team_result,result_points,cumulative_team_points,cumulative_opponent_points,cumulative_result_points,league_position,gw_team_points_rank,gw_opponent_points_rank
194,1,True,229876,Boom Boom Eagle FC,Sam,Thorpe,185622,Kernowfornia United,Scott,Renfree-Tuck,...,35,23,Win,3,58,35,3,1,1.0,7.0
191,1,True,320432,Some Solanke-Panky?,Alex,Turnbull,327410,Eleven Hags,Liam,Thaker,...,34,20,Win,3,54,34,3,2,2.0,8.0
193,1,True,285674,CripplingHannover96,Ryan,McIntyre,336507,Average,None,None,...,41,5,Win,3,46,41,3,3,3.0,5.0
192,1,True,291619,*Stratford Sardines*,Flat,7,328396,TAA very much,James,Mason,...,39,6,Win,3,45,39,3,4,4.0,6.0
190,1,True,185528,Easy Three Points,Jonathan,Ako,321053,Up the Ding,Matthew,Stent,...,32,1,Win,3,33,32,3,5,9.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,38,False,285674,CripplingHannover96,Ryan,McIntyre,328396,TAA very much,James,Mason,...,0,0,Draw,1,603,571,44,6,5.5,5.5
189,38,False,320432,Some Solanke-Panky?,Alex,Turnbull,321053,Up the Ding,Matthew,Stent,...,0,0,Draw,1,593,630,42,7,5.5,5.5
378,38,False,327410,Eleven Hags,Liam,Thaker,291619,*Stratford Sardines*,Flat,7,...,0,0,Draw,1,555,602,42,8,5.5,5.5
188,38,False,291619,*Stratford Sardines*,Flat,7,327410,Eleven Hags,Liam,Thaker,...,0,0,Draw,1,602,646,41,9,5.5,5.5


,team_id,entry_name,player_first_name,player_last_name,gw,number,position,total_points,minutes_played,first_name,second_name,web_name,element_type,team,status,Player Name,Price
0,NaN,NaN,NaN,NaN,1,1,NaN,0.0,no,Fábio,Ferreira Vieira,Fábio Vieira,3,1,benched,Fábio Ferreira Vieira,5.5
1,184679,Kernowfornia United,Scott,Renfree-Tuck,1,2,11.0,0.0,yes,Gabriel,Fernando de Jesus,G.Jesus,4,1,played,Gabriel Fernando de Jesus,7.0
2,317751,Some Solanke-Panky?,Alex,Turnbull,1,3,2.0,6.0,yes,Gabriel,dos Santos Magalhães,Gabriel,2,1,played,Gabriel dos Santos Magalhães,6.0
3,317751,Some Solanke-Panky?,Alex,Turnbull,1,4,10.0,12.0,yes,Kai,Havertz,Havertz,4,1,played,Kai Havertz,8.0
4,NaN,NaN,NaN,NaN,1,5,NaN,0.0,no,Karl,Hein,Hein,1,1,benched,Karl Hein,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26329,NaN,NaN,NaN,NaN,38,689,NaN,NaN,NaN,Treymaurice,Nyoni,Nyoni,3,12,benched,NaN,NaN
26330,NaN,NaN,NaN,NaN,38,690,NaN,NaN,NaN,Luca,Williams-Barnett,Williams-Barnett,3,18,benched,NaN,NaN
26331,NaN,NaN,NaN,NaN,38,691,NaN,NaN,NaN,Amara,Nallo,Nallo,2,12,benched,NaN,NaN
26332,NaN,NaN,NaN,NaN,38,692,NaN,NaN,NaN,Godwill,Kukonki,Kukonki,2,14,benched,NaN,NaN


,id,element_in,element_out,entry,event,kind,result,web_name_x,web_name_y,total_points_in,total_points_out
0,1,593,244,184679,1,w,di,De Ligt,Castagne,40.0,5.0
1,2,593,387,228518,1,w,di,De Ligt,Shaw,40.0,2.0
2,3,593,172,318371,1,w,a,De Ligt,James,40.0,5.0
3,4,231,387,228518,1,f,a,Mykolenko,Shaw,38.0,2.0
4,5,594,244,184679,1,f,a,Mazraoui,Castagne,44.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...
300,301,88,291,324701,15,f,a,Collins,Faes,7.0,1.0
301,302,572,88,324701,15,f,a,Yoro,Collins,1.0,7.0
302,303,358,508,318371,15,f,a,Ortega Moreno,Vicario,1.0,0.0
303,304,530,323,184585,15,f,a,Souček,Gravenberch,9.0,0.0


,choice_time,element,entry,entry_name,id_x,index,league,pick,player_first_name,player_last_name,...,element_type,team,position_points_rank,pick_order,position_pick_order,points_vs_position_pick,points_vs_position_pick_rank,all_players_position_points_rank,all_players_points_vs_position_pick,all_players_points_vs_position_pick_rank
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3,1,NaN,NaN,NaN,NaN,NaN,230.0,NaN,NaN
1,2024-08-13T19:40:31.989875Z,2,184679.0,Kernowfornia United,3320871.0,36.0,48142.0,9.0,Scott,Renfree-Tuck,...,4,1,23.0,36.0,11.0,-12.0,24.0,37.0,-26.0,24.0
2,2024-08-13T19:35:34.351939Z,3,317751.0,Some Solanke-Panky?,3320858.0,23.0,48142.0,5.0,Alex,Turnbull,...,2,1,3.0,23.0,5.0,2.0,18.0,3.0,2.0,12.0
3,2024-08-13T19:33:38.154723Z,4,317751.0,Some Solanke-Panky?,3320849.0,14.0,48142.0,5.0,Alex,Turnbull,...,4,1,9.0,14.0,5.0,-4.0,16.0,13.0,-8.0,16.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1,NaN,NaN,NaN,NaN,NaN,35.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3,12,NaN,NaN,NaN,NaN,NaN,313.0,NaN,NaN
689,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3,18,NaN,NaN,NaN,NaN,NaN,314.0,NaN,NaN
690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2,12,NaN,NaN,NaN,NaN,NaN,229.0,NaN,NaN
691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2,14,NaN,NaN,NaN,NaN,NaN,230.0,NaN,NaN


### Export Data to CSVs

In [87]:
### GW and Team
gw_and_team.to_csv(f'{folder}\\gw_and_team.csv', index=False)

### Match Results
match_results_ordered.to_csv(f'{folder}\\matches.csv', index=False)

### Player Scores
team_player_scores_with_cost.to_csv(f'{folder}\\player_scores.csv', index=False)

### Transfers Made
transfers.to_csv(f'{folder}\\transfers.csv', index=False)

### Originally Drafted Teams
draft_picks_df.to_csv(f'{folder}\\original_draft_picks.csv', index=False)